In [ ]:
# 🔄 ETL: DB2i CIDMAS → Doris Bronze Layer
# Source      : M3FDBPRD.CIDMAS (IBM i / AS400)
# Destination : bronze.cidmas  (Apache Doris)
# Pattern     : Extract → Transform (add metadata) → Load
#
# KEY FIXES vs original:
#   1. SparkSession.builder uses ONLY .appName().getOrCreate()
#      PYSPARK_SUBMIT_ARGS in docker-compose already owns master + jars.
#      Specifying .master() or .config("spark.jars") here creates a second
#      conflicting SparkContext → "Cannot call methods on a stopped SparkContext"
#
#   2. Removed the broken re-init catch block — it was masking Bug #1
#      and returning the same dead session via getOrCreate().
#
#   3. jt400 options updated to match the working test connection
#      (secure=false since Fix 1 succeeded — VPN handles encryption)
#
#   4. Removed the double .count() calls — count() is expensive on large
#      tables. Use it only when you genuinely need the number.
# =============================================================================

# %%
import os
from datetime import datetime
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit, trim
from pyspark.sql.types import DecimalType, StringType

# =============================================================================
# Step 0: Load .env (same helper as the test notebook)
# =============================================================================
def load_env_file(env_path="/home/jovyan/.env"):
    env_paths = [Path(env_path), Path(".env"), Path("/home/jovyan/work/.env")]
    for p in env_paths:
        if p.exists():
            with open(p) as f:
                for line in f:
                    line = line.strip()
                    if not line or line.startswith("#") or "=" not in line:
                        continue
                    key, value = line.split("=", 1)
                    if key.strip() not in os.environ:
                        os.environ[key.strip()] = value.strip()
            print(f"✅ .env loaded from {p.resolve()}")
            return
    print("⚠️  No .env file found — using environment variables only")

load_env_file()

# =============================================================================
# Step 1: Configuration
# =============================================================================
def load_config():
    return {
        "db2i": {
            # ── Connection ────────────────────────────────────────────────
            "url": (
                f"jdbc:as400://{os.getenv('DB2I_HOST', 'A400DEV.IbnSina-Pharma.Com')}"
                f":{os.getenv('DB2I_PORT', '8471')}"
                f";libraries={os.getenv('DB2I_SCHEMA', 'M3FDBPRD')};"
            ),
            "user":     os.getenv("DB2I_USER", ""),
            "password": os.getenv("DB2I_PASSWORD", ""),
            "driver":   "com.ibm.as400.access.AS400JDBCDriver",

            # ── Table & Partition ─────────────────────────────────────────
            # We query M3FDBPRD.CIDMAS — schema is already in the URL's libraries
            # so just the table name is enough. Use SCHEMA.TABLE to be explicit.
            "table":            "M3FDBPRD.CIDMAS",
            "partition_column": "IDCONO",
            "lower_bound":      os.getenv("DB2I_IDCONO_MIN", "0"),
            "upper_bound":      os.getenv("DB2I_IDCONO_MAX", "999"),
            "num_partitions":   int(os.getenv("DB2I_NUM_PARTITIONS", "3")),

            # ── jt400 driver options ──────────────────────────────────────
            # These are the WORKING options confirmed by the connection test.
            # "secure": "false" because VPN/FortiClient already encrypts the tunnel.
            # Port 8471 is plain DDM on this server — no TLS at the app layer.
            "extra_options": {
                "secure":           "false",  # ← VPN handles encryption
                "timeout":          "30",
                "socketTimeout":    "60",
                "prompt":           "false",  # no GUI sign-on popup
                "naming":           "sql",    # SCHEMA.TABLE format (not SCHEMA/TABLE)
                "translate binary": "true",   # correct binary field handling
            },
        },
        "doris": {
            "fenodes":    f"{os.getenv('DORIS_FE_HOST', 'host.docker.internal')}:{os.getenv('DORIS_FE_HTTP_PORT', '8030')}",
            "table":      "bronze.cidmas",
            "user":       os.getenv("DORIS_USER", "root"),
            "password":   os.getenv("DORIS_PASSWORD", ""),
            "format":     "json",
            "batch_size": int(os.getenv("DORIS_BATCH_SIZE", "50000")),
            "enable_2pc": os.getenv("DORIS_ENABLE_2PC", "false").lower() == "true",
        },
    }

cfg = load_config()
print(f"🔧 DB2i  : {cfg['db2i']['url']}")
print(f"🔧 Table : {cfg['db2i']['table']}")
print(f"🔧 Doris : {cfg['doris']['fenodes']} → {cfg['doris']['table']}")

# =============================================================================
# Step 2: SparkSession
# =============================================================================
# ✅ FIX #1 — DO NOT specify .master() or .config("spark.jars") here.
#
# Why: PYSPARK_SUBMIT_ARGS in docker-compose.yml already passes:
#   --master spark://spark-master:7077
#   --jars /jars/spark-doris-connector-spark-3.5-25.2.0.jar,/jars/jt400-21.0.6.jar
#
# Adding .master() or .config("spark.jars") here would try to create a SECOND
# SparkContext which kills the first one → "Cannot call methods on stopped SparkContext"
#
# Rule: PYSPARK_SUBMIT_ARGS owns cluster config. Builder only sets the app name.
# =============================================================================
print("\n🚀 Initializing SparkSession...")

spark = (
    SparkSession.builder
    .appName("db2i_cidmas_to_doris_bronze")
    # ── Timeouts: important for slow IBM i connections ─────────────────────
    .config("spark.network.timeout",              "800s")
    .config("spark.executor.heartbeatInterval",   "60s")
    .config("spark.sql.broadcastTimeout",         "600s")
    # ── Disable AQE to avoid "stopped SparkContext" in CoalesceShufflePartitions
    # AQE tries to optimize partitions at runtime but this interacts badly with
    # re-created SparkContexts. Safe to disable for JDBC-heavy ETL jobs.
    .config("spark.sql.adaptive.enabled",         "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"✅ Spark {spark.version} ready")
print(f"   Master    : {spark.sparkContext.master}")
print(f"   App ID    : {spark.sparkContext.applicationId}")
print(f"   Spark UI  : http://localhost:4041")

# Quick sanity check — simple local operation, no cluster needed
test = spark.range(1).count()
print(f"   Self-test : {test} ✓")

# =============================================================================
# Step 3: EXTRACT — Read from DB2i CIDMAS
# =============================================================================
print("\n" + "="*60)
print("📥 EXTRACT: Reading from DB2i M3FDBPRD.CIDMAS")
print("="*60)

# ── Build the JDBC reader ──────────────────────────────────────────────────
# dbtable uses a subquery alias WITHOUT "as" keyword
# DB2i / AS400 does not support the "as" keyword in subquery aliases
# ✅ CORRECT:   (SELECT * FROM table) tmp
# ❌ WRONG:     (SELECT * FROM table) AS tmp   ← SQL0104 on DB2i
reader = (
    spark.read
    .format("jdbc")
    .option("url",             cfg["db2i"]["url"])
    .option("dbtable",         f"(SELECT * FROM {cfg['db2i']['table']}) tmp")
    .option("user",            cfg["db2i"]["user"])
    .option("password",        cfg["db2i"]["password"])
    .option("driver",          cfg["db2i"]["driver"])
    .option("fetchsize",       "10000")
    # ── Parallel read options ──────────────────────────────────────────────
    # Spark will issue 3 parallel JDBC queries splitting IDCONO into ranges.
    # Each query runs on a separate worker partition.
    # numPartitions also = number of concurrent DB connections — don't set too high.
    .option("partitionColumn", cfg["db2i"]["partition_column"])
    .option("lowerBound",      cfg["db2i"]["lower_bound"])
    .option("upperBound",      cfg["db2i"]["upper_bound"])
    .option("numPartitions",   str(cfg["db2i"]["num_partitions"]))
)

# Apply jt400-specific options
for key, value in cfg["db2i"]["extra_options"].items():
    reader = reader.option(key, value)

# ── Execute the read (lazy — data is fetched only when an action is called) ──
db2i_df = reader.load()

# ⚠️  IMPORTANT: avoid .count() here on large tables — it reads ALL data twice.
# Use .limit(1).count() for a lightweight connection test instead.
print("✅ JDBC reader built — executing schema fetch...")
db2i_df.printSchema()

# Preview 2 rows (triggers actual data fetch — confirm connection works)
print("\n📊 Sample rows from DB2i:")
db2i_df.show(2, truncate=False)

# =============================================================================
# Step 4: TRANSFORM — Add Bronze Metadata
# =============================================================================
print("\n" + "="*60)
print("🔄 TRANSFORM: Casting columns + adding DWH metadata")
print("="*60)

# Batch ID: unique identifier for this pipeline run
# Useful for incremental loads, auditing, and rollback
batch_id = f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{spark.sparkContext.applicationId[-8:]}"

bronze_df = (
    db2i_df
    # ── Cast numeric columns ───────────────────────────────────────────────
    .withColumn("idcono", col("IDCONO").cast(DecimalType(3, 0)))
    .withColumn("idsuty", col("IDSUTY").cast(DecimalType(1, 0)))
    .withColumn("idpoda", col("IDPODA").cast(DecimalType(3, 0)))
    .withColumn("idcfi2", col("IDCFI2").cast(DecimalType(15, 2)))
    .withColumn("idtxid", col("IDTXID").cast(DecimalType(13, 0)))
    .withColumn("idrgdt", col("IDRGDT").cast(DecimalType(8, 0)))
    .withColumn("idrgtm", col("IDRGTM").cast(DecimalType(6, 0)))
    .withColumn("idlmdt", col("IDLMDT").cast(DecimalType(8, 0)))
    .withColumn("idchno", col("IDCHNO").cast(DecimalType(3, 0)))
    .withColumn("idppin", col("IDPPIN").cast(DecimalType(1, 0)))
    .withColumn("idlmts", col("IDLMTS").cast(DecimalType(18, 0)))
    .withColumn("idgeoc", col("IDGEOC").cast(DecimalType(9, 0)))
    # ── Cast + trim string columns ─────────────────────────────────────────
    # DB2i/AS400 stores fixed-length CHAR fields padded with spaces.
    # trim() removes the trailing spaces — critical for correct string matching.
    .withColumn("idsuno", trim(col("IDSUNO").cast(StringType())))
    .withColumn("idsunm", trim(col("IDSUNM").cast(StringType())))
    .withColumn("idalsu", trim(col("IDALSU").cast(StringType())))
    .withColumn("idstat", trim(col("IDSTAT").cast(StringType())))
    .withColumn("idcorg", trim(col("IDCORG").cast(StringType())))
    .withColumn("idcor2", trim(col("IDCOR2").cast(StringType())))
    .withColumn("idlncd", trim(col("IDLNCD").cast(StringType())))
    .withColumn("idphno", trim(col("IDPHNO").cast(StringType())))
    .withColumn("idphn2", trim(col("IDPHN2").cast(StringType())))
    .withColumn("idtlno", trim(col("IDTLNO").cast(StringType())))
    .withColumn("idtfno", trim(col("IDTFNO").cast(StringType())))
    .withColumn("idcscd", trim(col("IDCSCD").cast(StringType())))
    .withColumn("idecar", trim(col("IDECAR").cast(StringType())))
    .withColumn("iddtfm", trim(col("IDDTFM").cast(StringType())))
    .withColumn("idedit", trim(col("IDEDIT").cast(StringType())))
    .withColumn("idvrno", trim(col("IDVRNO").cast(StringType())))
    .withColumn("idsuco", trim(col("IDSUCO").cast(StringType())))
    .withColumn("idsual", trim(col("IDSUAL").cast(StringType())))
    .withColumn("idsucm", trim(col("IDSUCM").cast(StringType())))
    .withColumn("idmepf", trim(col("IDMEPF").cast(StringType())))
    .withColumn("idcfi1", trim(col("IDCFI1").cast(StringType())))
    .withColumn("idcfi3", trim(col("IDCFI3").cast(StringType())))
    .withColumn("idcfi4", trim(col("IDCFI4").cast(StringType())))
    .withColumn("idcfi5", trim(col("IDCFI5").cast(StringType())))
    .withColumn("idhafe", trim(col("IDHAFE").cast(StringType())))
    .withColumn("iddesv", trim(col("IDDESV").cast(StringType())))
    .withColumn("idtizo", trim(col("IDTIZO").cast(StringType())))
    .withColumn("idfwsc", trim(col("IDFWSC").cast(StringType())))
    .withColumn("idchid", trim(col("IDCHID").cast(StringType())))
    .withColumn("idcuex", trim(col("IDCUEX").cast(StringType())))
    .withColumn("idrasn", trim(col("IDRASN").cast(StringType())))
    # ── DWH metadata columns ───────────────────────────────────────────────
    # __dwh_load_timestamp : exact datetime this row was processed by Spark
    # __dwh_batch_id       : unique per pipeline run — use for audit/rollback
    # __dwh_source_system  : label showing where the data came from
    #.withColumn("__dwh_load_timestamp", current_timestamp())
    #.withColumn("__dwh_batch_id",       lit(batch_id))
    #.withColumn("__dwh_source_system",  lit("db2i_m3fdbprd"))

    .withColumn("dwh_load_date",        current_date())         # DATE column in Doris
    .withColumn("dwh_load_timestamp",   current_timestamp())    # DATETIME column in Doris  
    .withColumn("dwh_batch_id",         lit(batch_id))
    .withColumn("dwh_source_system",    lit("db2i_m3fdbprd"))

    
    # ── Final column selection (only the columns Doris table expects) ──────
    .select(
        # Numeric fields
        "idcono", "idsuty", "idpoda", "idcfi2", "idtxid",
        "idrgdt", "idrgtm", "idlmdt", "idchno", "idppin", "idlmts", "idgeoc",
        # String fields
        "idsuno", "idsunm", "idalsu", "idstat", "idcorg", "idcor2", "idlncd",
        "idphno", "idphn2", "idtlno", "idtfno", "idcscd", "idecar", "iddtfm",
        "idedit", "idvrno", "idsuco", "idsual", "idsucm", "idmepf", "idcfi1",
        "idcfi3", "idcfi4", "idcfi5", "idhafe", "iddesv", "idtizo", "idfwsc",
        "idchid", "idcuex", "idrasn",
        # DWH metadata
        #"__dwh_load_timestamp", "__dwh_batch_id", "__dwh_source_system",
        "dwh_load_date",       # ← matches Doris column name
        "dwh_load_timestamp",  # ← matches Doris column name
        "dwh_batch_id",
        "dwh_source_system",
    )
)

print("✅ Transform plan built (lazy — executes on write)")
print(f"   Batch ID : {batch_id}")
bronze_df.printSchema()

# =============================================================================
# Step 5: LOAD — Write to Apache Doris
# =============================================================================
print("\n" + "="*60)
print(f"📤 LOAD: Writing to Doris {cfg['doris']['table']}")
print("="*60)

(
    bronze_df.write
    .format("doris")
    .option("doris.table.identifier",              cfg["doris"]["table"])
    .option("doris.fenodes",                        cfg["doris"]["fenodes"])
    .option("user",                                 cfg["doris"]["user"])
    .option("password",                             cfg["doris"]["password"])
    # ── Write format ──────────────────────────────────────────────────────
    .option("doris.sink.properties.format",         cfg["doris"]["format"])
    .option("doris.sink.batch.size",                str(cfg["doris"]["batch_size"]))
    .option("doris.sink.enable-2pc",                str(cfg["doris"]["enable_2pc"]).lower())
    .option("doris.sink.auto-redirect",             "true")
    # ── Lenient mode for initial Bronze loads ─────────────────────────────
    # strict_mode=false : don't reject rows with type mismatches (log them instead)
    # max_filter_ratio  : allow up to 1% of rows to fail (for dirty source data)
    # Use strict_mode=true in production after validating the data profile
    .option("doris.sink.properties.strict_mode",    "false")
    .option("doris.sink.properties.max_filter_ratio","0.01")
    .option("doris.sink.properties.timeout",        "600")
    .mode("append")
    .save()
)

print(f"✅ Load complete!")
print(f"   Batch ID : {batch_id}")

# =============================================================================
# Step 6: VERIFY — Read back from Doris
# =============================================================================
print("\n" + "="*60)
print("🔍 VERIFY: Reading back from Doris")
print("="*60)

verify_df = (
    spark.read
    .format("doris")
    .option("doris.table.identifier", cfg["doris"]["table"])
    .option("doris.fenodes",           cfg["doris"]["fenodes"])
    .option("user",                    cfg["doris"]["user"])
    .option("password",                cfg["doris"]["password"])
    # Filter to ONLY this batch — avoids scanning all historical data
    .option("doris.filter.query",      f"__dwh_batch_id = '{batch_id}'")
    .load()
)

count = verify_df.count()
print(f"✅ Verified: {count} rows loaded in this batch")

if count > 0:
    print("\n📊 Sample from Doris:")
    (
        verify_df
        .select("idcono", "idsuno", "idsunm", "__dwh_load_timestamp", "__dwh_batch_id")
        .show(5, truncate=False)
    )
else:
    print("⚠️  Zero rows found — check Doris Stream Load error logs:")
    print(f"   mysql -h {os.getenv('DORIS_FE_HOST')} -P 9030 -u root")
    print("   SHOW LOAD FROM bronze ORDER BY CreateTime DESC LIMIT 5;")

# =============================================================================
# Step 7: Cleanup
# =============================================================================
# ⚠️  DO NOT call spark.stop() inside a notebook cell.
# Stopping the SparkContext kills it for ALL subsequent cells in the kernel.
# The kernel will restart it when needed — or you can restart the kernel manually.
#
# Only stop Spark in standalone scripts (not notebooks) or at the very end
# of a pipeline that will not run any more Spark code.
#
# spark.stop()  ← COMMENTED OUT intentionally

print("\n" + "="*60)
print("✨ ETL pipeline completed successfully")
print(f"   Source  : M3FDBPRD.CIDMAS  (IBM i / AS400)")
print(f"   Target  : bronze.cidmas    (Apache Doris)")
print(f"   Batch   : {batch_id}")
print("="*60)

In [ ]:
# Run this test cell first:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
print(f"✅ Spark {spark.version} ready")

In [ ]:
# check the jars
import os
jar_path = "/jars"
print(f"📁 Contents of {jar_path}:")
for f in sorted(os.listdir(jar_path)):
    print(f"   - {f}")

In [1]:
# 🔄 ETL: DB2i CIDMAS → Doris Bronze Layer v3 -- worked
# Source      : M3FDBPRD.CIDMAS (IBM i / AS/400)
# Destination : bronze.cidmas  (Apache Doris)
# Pattern     : Extract → Transform (add metadata) → Load
#
# KEY FIXES:
#   1. Removed dwh_load_timestamp (not in Doris DDL)
#   2. Fixed column order to match Doris DDL exactly (DUPLICATE KEY first!)
#   3. Updated doris.write.fields to match SELECT order
# =============================================================================
# %%
import os
from datetime import datetime
from pathlib import Path
from pyspark.sql import SparkSession
# ✅ FIX: Added current_date to imports
from pyspark.sql.functions import col, current_date, lit, trim
from pyspark.sql.types import DecimalType, StringType

# =============================================================================
# Step 0: Load .env
# =============================================================================
def load_env_file(env_path="/home/jovyan/.env"):
    env_paths = [Path(env_path), Path(".env"), Path("/home/jovyan/work/.env")]
    for p in env_paths:
        if p.exists():
            with open(p) as f:
                for line in f:
                    line = line.strip()
                    if not line or line.startswith("#") or "=" not in line:
                        continue
                    key, value = line.split("=", 1)
                    if key.strip() not in os.environ:
                        os.environ[key.strip()] = value.strip()
            print(f"✅ .env loaded from {p.resolve()}")
            return
    print("⚠️  No .env file found — using environment variables only")

load_env_file()

# =============================================================================
# Step 1: Configuration
# =============================================================================
def load_config():
    return {
        "db2i": {
            "url": (
                f"jdbc:as400://{os.getenv('DB2I_HOST', 'A400DEV.IbnSina-Pharma.Com')}"
                f":{os.getenv('DB2I_PORT', '8471')}"
                f";libraries={os.getenv('DB2I_SCHEMA', 'M3FDBPRD')};"
            ),
            "user":     os.getenv("DB2I_USER", ""),
            "password": os.getenv("DB2I_PASSWORD", ""),
            "driver":   "com.ibm.as400.access.AS400JDBCDriver",
            "table":            "M3FDBPRD.CIDMAS",
            "partition_column": "IDCONO",
            "lower_bound":      os.getenv("DB2I_IDCONO_MIN", "0"),
            "upper_bound":      os.getenv("DB2I_IDCONO_MAX", "999"),
            "num_partitions":   int(os.getenv("DB2I_NUM_PARTITIONS", "3")),
            "extra_options": {
                "secure":           "false",
                "timeout":          "30",
                "socketTimeout":    "60",
                "prompt":           "false",
                "naming":           "sql",
                "translate binary": "true",
            },
        },
        "doris": {
            "fenodes":    f"{os.getenv('DORIS_FE_HOST', 'host.docker.internal')}:{os.getenv('DORIS_FE_HTTP_PORT', '8030')}",
            "table":      "bronze.cidmas",
            "user":       os.getenv("DORIS_USER", "root"),
            "password":   os.getenv("DORIS_PASSWORD", ""),
            "format":     "json",
            "batch_size": int(os.getenv("DORIS_BATCH_SIZE", "50000")),
            "enable_2pc": os.getenv("DORIS_ENABLE_2PC", "false").lower() == "true",
        },
    }

cfg = load_config()
print(f"🔧 DB2i  : {cfg['db2i']['url']}")
print(f"🔧 Table : {cfg['db2i']['table']}")
print(f"🔧 Doris : {cfg['doris']['fenodes']} → {cfg['doris']['table']}")

# =============================================================================
# Step 2: SparkSession
# =============================================================================
print("🚀 Initializing SparkSession...")
spark = (
    SparkSession.builder
    .appName("db2i_cidmas_to_doris_bronze")
    .config("spark.network.timeout",              "800s")
    .config("spark.executor.heartbeatInterval",   "60s")
    .config("spark.sql.broadcastTimeout",         "600s")
    .config("spark.sql.adaptive.enabled",         "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark {spark.version} ready")
print(f"   Master    : {spark.sparkContext.master}")
print(f"   App ID    : {spark.sparkContext.applicationId}")

test = spark.range(1).count()
print(f"   Self-test : {test} ✓")

# =============================================================================
# Step 3: EXTRACT — Read from DB2i CIDMAS
# =============================================================================
print("\n" + "="*60)
print("📥 EXTRACT: Reading from DB2i M3FDBPRD.CIDMAS")
print("="*60)

reader = (
    spark.read
    .format("jdbc")
    .option("url",             cfg["db2i"]["url"])
    .option("dbtable",         f"(SELECT * FROM {cfg['db2i']['table']}) tmp")
    .option("user",            cfg["db2i"]["user"])
    .option("password",        cfg["db2i"]["password"])
    .option("driver",          cfg["db2i"]["driver"])
    .option("fetchsize",       "10000")
    .option("partitionColumn", cfg["db2i"]["partition_column"])
    .option("lowerBound",      cfg["db2i"]["lower_bound"])
    .option("upperBound",      cfg["db2i"]["upper_bound"])
    .option("numPartitions",   str(cfg["db2i"]["num_partitions"]))
)

for key, value in cfg["db2i"]["extra_options"].items():
    reader = reader.option(key, value)

db2i_df = reader.load()

print("✅ JDBC reader built — executing schema fetch...")
db2i_df.printSchema()
print("\n📊 Sample rows from DB2i:")
db2i_df.show(2, truncate=False)

# =============================================================================
# Step 4: TRANSFORM — Add Bronze Metadata
# =============================================================================
print("\n" + "="*60)
print("🔄 TRANSFORM: Casting columns + adding DWH metadata")
print("="*60)

batch_id = f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{spark.sparkContext.applicationId[-8:]}"

# ✅ FIX: Column order MUST match Doris DDL exactly!
# Doris DDL: idcono, idsuno, idsuty, idsunm, idalsu, idstat, ...
# DUPLICATE KEY columns (idcono, idsuno) MUST be first!
bronze_df = (
    db2i_df
    # ── Cast numeric columns ───────────────────────────────────────────────
    .withColumn("idcono", col("IDCONO").cast(DecimalType(3, 0)))
    .withColumn("idsuty", col("IDSUTY").cast(DecimalType(1, 0)))
    .withColumn("idpoda", col("IDPODA").cast(DecimalType(3, 0)))
    .withColumn("idcfi2", col("IDCFI2").cast(DecimalType(15, 2)))
    .withColumn("idtxid", col("IDTXID").cast(DecimalType(13, 0)))
    .withColumn("idrgdt", col("IDRGDT").cast(DecimalType(8, 0)))
    .withColumn("idrgtm", col("IDRGTM").cast(DecimalType(6, 0)))
    .withColumn("idlmdt", col("IDLMDT").cast(DecimalType(8, 0)))
    .withColumn("idchno", col("IDCHNO").cast(DecimalType(3, 0)))
    .withColumn("idppin", col("IDPPIN").cast(DecimalType(1, 0)))
    .withColumn("idlmts", col("IDLMTS").cast(DecimalType(18, 0)))
    .withColumn("idgeoc", col("IDGEOC").cast(DecimalType(9, 0)))
    # ── Cast + trim string columns ─────────────────────────────────────────
    .withColumn("idsuno", trim(col("IDSUNO").cast(StringType())))
    .withColumn("idsunm", trim(col("IDSUNM").cast(StringType())))
    .withColumn("idalsu", trim(col("IDALSU").cast(StringType())))
    .withColumn("idstat", trim(col("IDSTAT").cast(StringType())))
    .withColumn("idcorg", trim(col("IDCORG").cast(StringType())))
    .withColumn("idcor2", trim(col("IDCOR2").cast(StringType())))
    .withColumn("idlncd", trim(col("IDLNCD").cast(StringType())))
    .withColumn("idphno", trim(col("IDPHNO").cast(StringType())))
    .withColumn("idphn2", trim(col("IDPHN2").cast(StringType())))
    .withColumn("idtlno", trim(col("IDTLNO").cast(StringType())))
    .withColumn("idtfno", trim(col("IDTFNO").cast(StringType())))
    .withColumn("idcscd", trim(col("IDCSCD").cast(StringType())))
    .withColumn("idecar", trim(col("IDECAR").cast(StringType())))
    .withColumn("iddtfm", trim(col("IDDTFM").cast(StringType())))
    .withColumn("idedit", trim(col("IDEDIT").cast(StringType())))
    .withColumn("idvrno", trim(col("IDVRNO").cast(StringType())))
    .withColumn("idsuco", trim(col("IDSUCO").cast(StringType())))
    .withColumn("idsual", trim(col("IDSUAL").cast(StringType())))
    .withColumn("idsucm", trim(col("IDSUCM").cast(StringType())))
    .withColumn("idmepf", trim(col("IDMEPF").cast(StringType())))
    .withColumn("idcfi1", trim(col("IDCFI1").cast(StringType())))
    .withColumn("idcfi3", trim(col("IDCFI3").cast(StringType())))
    .withColumn("idcfi4", trim(col("IDCFI4").cast(StringType())))
    .withColumn("idcfi5", trim(col("IDCFI5").cast(StringType())))
    .withColumn("idhafe", trim(col("IDHAFE").cast(StringType())))
    .withColumn("iddesv", trim(col("IDDESV").cast(StringType())))
    .withColumn("idtizo", trim(col("IDTIZO").cast(StringType())))
    .withColumn("idfwsc", trim(col("IDFWSC").cast(StringType())))
    .withColumn("idchid", trim(col("IDCHID").cast(StringType())))
    .withColumn("idcuex", trim(col("IDCUEX").cast(StringType())))
    .withColumn("idrasn", trim(col("IDRASN").cast(StringType())))
    # ── DWH metadata columns ───────────────────────────────────────────────
    # ✅ FIX: Only dwh_load_date (NOT dwh_load_timestamp - not in Doris DDL!)
    .withColumn("dwh_load_date",        current_date())
    .withColumn("dwh_batch_id",         lit(batch_id))
    .withColumn("dwh_source_system",    lit("db2i_m3fdbprd"))
    # ── Final column selection (MUST match Doris DDL order exactly!) ──────
    # ✅ DUPLICATE KEY columns FIRST: idcono, idsuno
    .select(
        # DUPLICATE KEY columns (MUST be first!)
        col("idcono"), col("idsuno"),
        # Remaining columns in Doris DDL order
        col("idsuty"), col("idsunm"), col("idalsu"), col("idstat"), col("idcorg"), col("idcor2"), col("idlncd"),
        col("idphno"), col("idphn2"), col("idtlno"), col("idtfno"), col("idcscd"), col("idecar"), col("iddtfm"),
        col("idedit"), col("idvrno"), col("idsuco"), col("idsual"), col("idsucm"), col("idmepf"), col("idpoda"),
        col("idcfi1"), col("idcfi2"), col("idcfi3"), col("idcfi4"), col("idcfi5"), col("idhafe"), col("iddesv"),
        col("idtizo"), col("idfwsc"), col("idtxid"), col("idrgdt"), col("idrgtm"), col("idlmdt"), col("idchno"),
        col("idchid"), col("idppin"), col("idlmts"), col("idgeoc"), col("idcuex"), col("idrasn"),
        # DWH metadata (exact names from Doris DDL)
        col("dwh_load_date"), col("dwh_batch_id"), col("dwh_source_system"),
    )
)

print("✅ Transform plan built (lazy — executes on write)")
print(f"   Batch ID : {batch_id}")
bronze_df.printSchema()

# =============================================================================
# Step 5: LOAD — Write to Apache Doris
# =============================================================================
print("\n" + "="*60)
print(f"📤 LOAD: Writing to Doris {cfg['doris']['table']}")
print("="*60)

(
    bronze_df.write
    .format("doris")
    .option("doris.table.identifier",              cfg["doris"]["table"])
    .option("doris.fenodes",                        cfg["doris"]["fenodes"])
    .option("user",                                 cfg["doris"]["user"])
    .option("password",                             cfg["doris"]["password"])
    .option("doris.sink.properties.format",         cfg["doris"]["format"])
    .option("doris.sink.batch.size",                str(cfg["doris"]["batch_size"]))
    .option("doris.sink.enable-2pc",                str(cfg["doris"]["enable_2pc"]).lower())
    .option("doris.sink.auto-redirect",             "true")
    .option("doris.sink.properties.strict_mode",    "false")
    .option("doris.sink.properties.max_filter_ratio","0.01")
    .option("doris.sink.properties.timeout",        "600")
    # ✅ FIX: Column order MUST match SELECT order above exactly!
    .option("doris.write.fields", 
            "idcono,idsuno,idsuty,idsunm,idalsu,idstat,idcorg,idcor2,idlncd,idphno,"
            "idphn2,idtlno,idtfno,idcscd,idecar,iddtfm,idedit,idvrno,idsuco,idsual,"
            "idsucm,idmepf,idpoda,idcfi1,idcfi2,idcfi3,idcfi4,idcfi5,idhafe,iddesv,"
            "idtizo,idfwsc,idtxid,idrgdt,idrgtm,idlmdt,idchno,idchid,idppin,idlmts,"
            "idgeoc,idcuex,idrasn,dwh_load_date,dwh_batch_id,dwh_source_system")
    .mode("append")
    .save()
)

print(f"✅ Load complete!")
print(f"   Batch ID : {batch_id}")

# =============================================================================
# Step 6: VERIFY — Read back from Doris
# =============================================================================
print("\n" + "="*60)
print("🔍 VERIFY: Reading back from Doris")
print("="*60)

verify_df = (
    spark.read
    .format("doris")
    .option("doris.table.identifier", cfg["doris"]["table"])
    .option("doris.fenodes",           cfg["doris"]["fenodes"])
    .option("user",                    cfg["doris"]["user"])
    .option("password",                cfg["doris"]["password"])
    .option("doris.filter.query",      f"dwh_batch_id = '{batch_id}'")
    .load()
)

count = verify_df.count()
print(f"✅ Verified: {count} rows loaded in this batch")

if count > 0:
    print("\n📊 Sample from Doris:")
    (
        verify_df
        .select("idcono", "idsuno", "idsunm", "dwh_load_date", "dwh_batch_id")
        .show(5, truncate=False)
    )
else:
    print("⚠️  Zero rows found — check Doris Stream Load error logs:")
    print(f"   mysql -h {os.getenv('DORIS_FE_HOST')} -P 9030 -u root")
    print("   SHOW LOAD FROM bronze ORDER BY CreateTime DESC LIMIT 5;")

# =============================================================================
# Step 7: Cleanup
# =============================================================================
print("\n" + "="*60)
print("✨ ETL pipeline completed successfully")
print(f"   Source  : M3FDBPRD.CIDMAS  (IBM i / AS/400)")
print(f"   Target  : bronze.cidmas    (Apache Doris)")
print(f"   Batch   : {batch_id}")
print("="*60)

✅ .env loaded from /home/jovyan/.env
🔧 DB2i  : jdbc:as400://A400DEV.IbnSina-Pharma.Com:8471;libraries=M3FDBPRD;
🔧 Table : M3FDBPRD.CIDMAS
🔧 Doris : host.docker.internal:8030 → bronze.cidmas
🚀 Initializing SparkSession...
✅ Spark 3.5.0 ready
   Master    : spark://spark-master:7077
   App ID    : app-20260331092525-0034
   Self-test : 1 ✓

📥 EXTRACT: Reading from DB2i M3FDBPRD.CIDMAS
✅ JDBC reader built — executing schema fetch...
root
 |-- IDCONO: decimal(3,0) (nullable = true)
 |-- IDSUNO: string (nullable = true)
 |-- IDSUTY: decimal(1,0) (nullable = true)
 |-- IDSUNM: string (nullable = true)
 |-- IDALSU: string (nullable = true)
 |-- IDSTAT: string (nullable = true)
 |-- IDCORG: string (nullable = true)
 |-- IDCOR2: string (nullable = true)
 |-- IDLNCD: string (nullable = true)
 |-- IDPHNO: string (nullable = true)
 |-- IDPHN2: string (nullable = true)
 |-- IDTLNO: string (nullable = true)
 |-- IDTFNO: string (nullable = true)
 |-- IDCSCD: string (nullable = true)
 |-- IDECAR: stri